# SMR multilink modelization


Combined Heat and Power (CHP) with fixed heat-power ratio. 

the following code is based in a PyPSA CHP multilink [example](https://pypsa.readthedocs.io/en/latest/examples/chp-fixed-heat-power-ratio.html) that demonstrates a Link component with more than one bus output (“bus2” in this case).

The SMR_CHP must be heat-following because there is no other supply of heat to the bus "Industry Electricity heat”.

In [ ]:
import pypsa

In [6]:
n0 = pypsa.Network() # Creates empty PyPSA network object called n0

n0.add("Bus", "Industry Electricity", carrier="AC")
n0.add("Load", "Industry Electricity Load", bus="Industry Electricity", p_set=5)


n0.add("Bus", "Industry Heat", carrier="Heat")
n0.add("Load", "Industry Heat Load", bus="Industry Heat", p_set=3)


n0.add("Bus", "Nuclear_Fuel", carrier="Uranium")
n0.add("Store", "Nuclear_Fuel", 
       e_initial=1e6, 
       e_nom=1e6, 
       bus="Nuclear_Fuel")

n0.add("Bus", "Spain gas", carrier="gas")
n0.add("Store", "Spain gas", 
       e_initial=1e6,   # Energy before the snapshots in the OPF (Optimized power Flow).
       e_nom=1e6,       # Nominal energy capacity
       bus="Spain gas")

n0.add(
    "Link",
    "OCGT",                 # Open Cycle Gas Turbine
    bus0="Spain gas",
    bus1="Industry Electricity",       # Only produces electricity
    p_nom_extendable=True,
    capital_cost=600,
    efficiency=0.4,
)

n0.add(
    "Link",
    "SMR_CHP",
    bus0="Nuclear_Fuel",
    bus1="Industry Electricity",
    bus2="Industry Heat",
    p_nom_extendable=True,
    capital_cost=14000,
    efficiency=0.35,
    efficiency2=0.35,
)

Index(['SMR_CHP'], dtype='object')

In [7]:
n0 #view the network n0

Unnamed PyPSA Network
---------------------
Components:
 - Bus: 4
 - Link: 2
 - Load: 2
 - Store: 2
Snapshots: 1

In [8]:
n0.buses.index

Index(['Industry Electricity', 'Industry Heat', 'Nuclear_Fuel', 'Spain gas'], dtype='object', name='Bus')

There are no generators because they have been modelled as links and buses.

In [11]:
# n0.generators.index 

In [12]:
n0.storage_units.index

RangeIndex(start=0, stop=0, step=1, name='StorageUnit')

In [13]:
n0.optimize();

Index(['Industry Electricity', 'Industry Heat', 'Nuclear_Fuel', 'Spain gas'], dtype='object', name='Bus')


Index(['OCGT', 'SMR_CHP'], dtype='object', name='Link')
Index(['Nuclear_Fuel', 'Spain gas'], dtype='object', name='Store')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.05s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 8 primals, 16 duals
Objective: 1.23e+05
Solver model: available
Solver message: optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Link-ext-p-lower, Link-ext-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


In [24]:
n0.loads_t.p

Load,Industry Electricity Load,Industry Heat Load
snapshot,,
now,5.0,3.0


Note that negative values for n0.links_t.p1 indicate that the link is injecting power into bus1 (This aligns with PyPSA's [convention](https://pypsa.readthedocs.io/en/latest/user-guide/design.html#sign-conventions) where power injected into a bus is considered negative, while power withdrawn from a bus is positive). 

- n0.links_t.p0 # energy flows through the input bus (bus0) in each link (for each snapshot) CONSUMO DE COMBUSTIBLE
- n0.links_t.p1 # energy flows through the output bus1 (bus1) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)
- n0.links_t.p2 # energy flows through the output bus2 (bus2) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)

<!-- Note that `negative values` for n0.links_t.p1 indicate that the link is injecting power into bus1 (This aligns with PyPSA's [convention](https://pypsa.readthedocs.io/en/latest/user-guide/design.html#sign-conventions) where power injected into a bus is considered negative, while power withdrawn from a bus is positive).

Here there is a brief explanation for each of the following code lines: 
- n0.links_t.p0: `energy flows through the input bus (bus0) in each link (for each snapshot) CONSUMO DE COMBUSTIBLE`
- n0.links_t.p1: `energy flows through the output bus1 (bus1) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)`
- n0.links_t.p2: `energy flows through the output bus2 (bus2) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)` -->

In [21]:
print("SMR Electricity Output:", n0.links_t.p1["SMR_CHP"].sum())  # -70 MW
print("SMR Heat Output:", n0.links_t.p2["SMR_CHP"].sum())         # -108 MW
print("SMR Uranium Fuel Consumption:", n0.links_t.p0["SMR_CHP"].sum())    # +314.3 MW

SMR Electricity Output: -2.9999999999999996
SMR Heat Output: -2.9999999999999996
SMR Uranium Fuel Consumption: 8.571428571428571


In [22]:
print("OCGT Electricity Output:", n0.links_t.p1["OCGT"].sum())
print("OCGT Gas Consumption:", n0.links_t.p0["OCGT"].sum())

OCGT Electricity Output: -2.0
OCGT Gas Consumption: 5.0


# End of the fuctional example

<!-- 
Starts SMR model versión2 with network `n2`

## Note: 
About de network initialization (can be useful for some future research)

**Network Initialization**: `n1 = pypsa.Network(url)`
   - Creates a PyPSA `Network` object `n1`
   - Automatically downloads and loads the pre-configured energy system model containing:
     - Buses (energy nodes)
     - Generators (power plants)
     - Loads (energy demands)
     - Transmission lines
     - Time-dependent constraints

Key Features of This Approach:
- **Reproducibility**: Loads a standardized benchmark model
- **Time-Saving**: Avoids manual network construction (~1000+ lines of setup code)
- **Research-Ready**: Contains realistic European power system data (likely from [PyPSA-Eur](https://pypsa-eur.readthedocs.io/)) -->